# 06. 71786 터널 사진 고르기 · 정답 만들기 (EXP-67 준비)

**하는 일**: AI Hub 71786 라벨에서 터널이 표시된 사진을 골라, 받을 사진 목록을 만들고, 사진이 오면 입구/출구를 나누고 천리안 정답을 붙여요.
결과 `data/aihub_71786/photos.csv` → 07 노트북의 "처음 보는 카메라 시험"(71786 학습 → Mapillary 시험, **주 지표**)에 들어가요.

**순서**
1. 라벨 6개에서 터널 사진 목록 (6,869장 예상)
2. 장면 묶기 (연달아 2분 · 1 km 안 = 한 장면) → 장면당 최대 3장 고르기
3. 받을 사진 목록 저장 (형이 AI Hub 원천데이터에서 받기)
4. 사진이 오면: 입구/출구 자동 분류 (터널 구멍이 주변보다 어두우면 입구)
5. 천리안 일사량 + ASOS 구름 붙이기 (04 노트북과 같은 방식)
6. `photos.csv` 저장

실행 위치: `sunsight_video` 폴더. 71786은 팀 내부용 데이터 (밖으로 공유 금지).
정리 문서: `sunsight_plan/모델3_실험정리/EXP-67_처음보는카메라_71786학습.md`

In [ ]:
import os, json, math, time, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd

DATA = Path('data')
SD = Path('..') / 'sunsight_data'
LAB = SD / 'data' / '71786_라벨'
A71 = DATA / 'aihub_71786'; (A71 / 'photos').mkdir(parents=True, exist_ok=True)
print('라벨 파일:', [p.name for p in sorted(LAB.glob('*.zip'))])

## 1. 터널 사진 목록
라벨 JSON의 `categories`에서 이름이 "터널"인 번호(=20)를 찾고, 그 번호 표시가 하나라도 있는 사진만 남겨요.

In [ ]:
rows = []
for p in sorted(LAB.glob('*.zip')):
    z = zipfile.ZipFile(p); src = p.stem
    for n in z.namelist():
        if not n.endswith('.json'): continue
        j = json.loads(z.read(n).decode('utf-8-sig'))
        tid = [c['id'] for c in j['categories'] if c['name'] == '터널']
        k = sum(1 for a in j['annotations'] if a.get('category_id') in tid)
        if k:
            im = j['images'][0]
            rows.append({'source_zip': src, 'json_path': n, 'file_name': im['file_name'], 'day': im['day'], 'time': im['time'],
                         'lat': im['latitude'], 'lon': im['longitude'], 'climate': im['climate'], 'camera': im['camera'],
                         'width': im['width'], 'height': im['height'], 'n_tunnel_poly': k})
lst = pd.DataFrame(rows)
lst['road'] = lst.file_name.str.split('_').str[0]
lst['captured_at'] = pd.to_datetime('20' + lst.day.astype(str) + ' ' + lst.time.astype(str))   # 한국시간
print(len(lst), '장'); print(lst.groupby(['source_zip']).size().to_dict())
print('날씨', lst.climate.value_counts().to_dict(), '| 카메라', lst.camera.value_counts().to_dict())

## 2. 장면 묶기 · 고르기
- 같은 라벨 파일 안에서 시간순으로 놓고, 앞 사진과 **2분 넘게** 떨어지거나 **1 km 넘게** 떨어지면 새 장면
- 연달아 찍은 사진은 거의 같은 사진 → 장면당 처음 · 가운데 · 끝 **최대 3장**만 (받을 양도 줄고, 한 장면이 학습을 덮지 않음)
- 고속도로 · 국도 모두 고르고, 07에서 "고속도로만 / 전부" 두 번 비교

In [ ]:
lst = lst.sort_values(['source_zip', 'captured_at']).reset_index(drop=True)
gap = lst.groupby('source_zip').captured_at.diff().dt.total_seconds()
dist = np.hypot(lst.lat.diff(), lst.lon.diff() * np.cos(np.radians(lst.lat))) * 111000
lst['scene'] = ((gap.isna()) | (gap > 120) | (dist > 1000)).cumsum() + 100000   # Mapillary 장면번호와 안 겹치게
def pick(g):
    i = sorted(set([0, len(g) // 2, len(g) - 1])); return g.iloc[i]
sel = lst.groupby('scene', group_keys=False).apply(pick)
print('장면', lst.scene.nunique(), '| 고른 사진', len(sel), '장', sel.road.value_counts().to_dict())
lst.to_csv(A71 / '터널사진_목록.csv', index=False, encoding='utf-8-sig')

## 3. 받을 사진 목록
- `받을사진_목록.csv` = 고른 사진 파일 이름과 들어 있는 라벨 묶음(TL_1 … VL_2). 원천데이터는 라벨과 같은 이름 규칙(TS_ / VS_)으로 나뉘어 있을 것
- 받은 사진은 `data/aihub_71786/photos/`에 파일 이름 그대로 넣기 (하위 폴더 없이)
- 원천 묶음이 너무 크면 **VS_1 고속도로(검증용)부터** 받아도 시작 가능 (터널 사진 651장)

In [ ]:
need = sel[['source_zip', 'file_name', 'road', 'captured_at']].copy()
need['원천_예상'] = need.source_zip.str.replace('TL_', 'TS_').str.replace('VL_', 'VS_')
# 터널 안 보이는 일반 도로 사진 (학습에만 씀, 11/5): 고속도로 · 낮 · 터널 프레임 ±4분 밖 · 1 km×하루 1장 · 시간대 고르게
road_path = A71 / '일반도로_고른목록.csv'
road_sel = pd.DataFrame()
if road_path.exists():
    road_sel = pd.read_csv(road_path, dtype={'day': str})
    road_sel['captured_at'] = pd.to_datetime('20' + road_sel.day + ' ' + road_sel.time)
    road_sel['source_zip'] = road_sel['원천'].str.replace('TS_', 'TL_').str.replace('VS_', 'VL_'); road_sel['원천_예상'] = road_sel['원천']
    road_sel['road'] = '고속도로'; road_sel['scene'] = range(300000, 300000 + len(road_sel))   # 한 장 = 한 장면
    need = pd.concat([need, road_sel[['source_zip', 'file_name', 'road', 'captured_at', '원천_예상']]], ignore_index=True)
    print('일반 도로 사진', len(road_sel), '장 추가 (학습용)')
need.to_csv(A71 / '받을사진_목록.csv', index=False, encoding='utf-8-sig')
print(need.groupby('원천_예상').size())
have = {p.name for p in (A71 / 'photos').glob('*.jpg')}
print('이미 받은 사진', len(have & set(need.file_name)), '/', len(need))

### 3-1. 받은 원천에서 필요한 사진만 꺼내기
- 원천 파일을 **풀지 말고** 그대로 두기 (지금: `sunsight_data/data/`). 이름이 `Unconfirmed ….crdownload`여도 크기가 다 찼으면 정상
- 아래 셀이 파일 안에서 고른 사진만 `photos/`로 꺼냄 (VS_1 340장 ≈ 2분)
- 다 꺼낸 뒤 원천 파일은 지워도 됨

In [ ]:
# AI Hub 원천은 "tar 안에 1 GB씩 쪼갠 zip" 모양 (이름이 .crdownload여도 다 받았으면 정상)
# → 풀지 않고 tar 안 조각들을 이어 붙인 것처럼 읽어서, 고른 사진만 꺼냄
import io, tarfile
SRC = SD / 'data'                      # 원천 파일을 둔 폴더. 다른 곳이면 바꾸기
class Cat(io.RawIOBase):
    def __init__(s, path, segs): s.f = open(path, 'rb'); s.segs = segs; s.size = sum(l for _, l in segs); s.pos = 0
    def seekable(s): return True
    def readable(s): return True
    def tell(s): return s.pos
    def seek(s, o, w=0): s.pos = o if w == 0 else s.pos + o if w == 1 else s.size + o; return s.pos
    def readinto(s, b):
        n, out, p, acc = len(b), 0, s.pos, 0
        for off, l in s.segs:
            if p < acc + l and out < n:
                k = min(n - out, acc + l - p); s.f.seek(off + (p - acc)); d = s.f.read(k)
                b[out:out + len(d)] = d; out += len(d); p += len(d)
            acc += l
        s.pos = p; return out
def open_source(path):
    if tarfile.is_tarfile(path):
        ms = sorted([m for m in tarfile.open(path).getmembers() if m.isfile() and '.zip.part' in m.name],
                    key=lambda m: int(m.name.rsplit('.zip.part', 1)[1]))
        print(Path(path).name, '→', ms[0].name.split('/')[-1].split('.zip')[0], len(ms), '조각')
        return zipfile.ZipFile(io.BufferedReader(Cat(path, [(m.offset_data, m.size) for m in ms]), buffer_size=1 << 20))
    return zipfile.ZipFile(path)

want = set(need.file_name); got = 0
cands = [p for p in SRC.iterdir() if p.is_file() and p.stat().st_size > 500e6]   # 큰 파일만 (원천)
for zp in cands:
    try: z = open_source(zp)
    except Exception as e: print('못 읽음:', zp.name, e); continue
    for n in z.namelist():
        fn = Path(n).name
        q = A71 / 'photos' / fn
        if fn in want and not (q.exists() and q.stat().st_size > 0):   # 중간에 끊겨 0바이트로 남은 사진은 다시
            (A71 / 'photos' / fn).write_bytes(z.read(n)); got += 1
have = {p.name for p in (A71 / 'photos').glob('*.jpg')}
print('새로 꺼냄', got, '| 지금 있는 사진', len(have & want), '/', len(want))
print(need.assign(있음=need.file_name.isin(have)).groupby('원천_예상').있음.agg(['sum', 'size']))

## 4. 입구/출구 자동 분류 (사진이 있을 때만)
- 터널 다각형 안 평균 밝기 vs 바로 바깥 띠(41칸 넓힘) 평균 밝기
- 71786 다각형은 입구 테두리 · 조명 켜진 안쪽까지 덮음 → 입구 사진도 안이 조금 밝을 수 있음
- 그래서 **둘레가 안보다 2.5배 이상 어두울 때만 출구**(터널 안에서 밖을 봄), 나머지는 입구. VS_1 340장은 전부 입구로 나옴 (11/5)
- 자동 분류라 틀릴 수 있음 → 아래 표본 30장을 눈으로 확인해서 맞은 비율을 EXP-67에 적기

In [ ]:
import cv2
polys = {}
want = set(sel.file_name) & {p.name for p in (A71 / 'photos').glob('*.jpg')}
for p in sorted(LAB.glob('*.zip')):
    z = zipfile.ZipFile(p)
    for n in z.namelist():
        fn = Path(n).stem + '.jpg'
        if fn not in want: continue
        j = json.loads(z.read(n).decode('utf-8-sig'))
        tid = [c['id'] for c in j['categories'] if c['name'] == '터널']
        polys[fn] = [a['segmentation'] for a in j['annotations'] if a.get('category_id') in tid]

def portal_class(fn):
    img = cv2.imdecode(np.fromfile(str(A71 / 'photos' / fn), np.uint8), cv2.IMREAD_GRAYSCALE)
    if img is None or fn not in polys: return None, np.nan
    mask = np.zeros(img.shape, np.uint8)
    for s in polys[fn]:
        cv2.fillPoly(mask, [np.array(s, float).reshape(-1, 2).astype(np.int32)], 1)
    ring = cv2.dilate(mask, np.ones((41, 41), np.uint8)).astype(bool) & ~mask.astype(bool)
    inside, around = img[mask.astype(bool)].mean(), img[ring].mean()
    r = np.log10((around + 1) / (inside + 1))
    # 71786 다각형은 입구 테두리(콘크리트) + 조명 켜진 터널 안까지 덮어서, 입구 사진도 안쪽이 주변보다 밝을 수 있음 (11/5 눈 확인)
    # 출구 = 터널 안에서 밝은 바깥을 보는 사진 → 둘레(터널 벽)가 훨씬 어두움. 2.5배 이상 차이일 때만 출구로
    return ('출구' if r < -0.4 else '입구'), r

cls = pd.DataFrame([(fn, *portal_class(fn)) for fn in sorted(want)], columns=['file_name', 'class', 'portal_log_ratio'])
print(cls['class'].value_counts().to_dict())
cls.sample(min(30, len(cls)), random_state=0).to_csv(A71 / '입구출구_눈확인_30장.csv', index=False, encoding='utf-8-sig')

## 5. 천리안 일사량 + ASOS 구름
- 04 노트북과 같은 API · 같은 규칙 (촬영 다음 정시 1시간 값). 요청은 **장면마다 1번** (같은 자리 · 같은 시각)
- 받아 둔 파일은 `data/gk2a_photo/`에 저장 → 다시 실행하면 건너뜀
- 구름(전운량)은 가장 가까운 ASOS 관측소 값 (규칙4 · 5 · LightGBM+ 입력). 71786은 라벨이 전부 "맑음"이라 실제 일사량이 낮은 사진이 있는지 여기서 확인

In [ ]:
import requests
CACHE = DATA / 'gk2a_photo'; CACHE.mkdir(parents=True, exist_ok=True)
BASE_URL = 'https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph_sun_sat_ana_txt'
def env(key):
    for line in (SD / '.env').read_text(encoding='utf-8-sig').splitlines():
        if line.strip().startswith(key + '='): return line.split('=', 1)[1].strip().strip('"').strip("'")
    return os.environ.get(key)
AUTH_KEY = env('KMA_APIHUB_KEY'); assert AUTH_KEY, 'sunsight_data/.env 에 KMA_APIHUB_KEY 필요'

ph = sel.merge(cls, on='file_name', how='inner') if len(cls) else sel.assign(**{'class': None})
if len(road_sel):
    have_now = {p.name for p in (A71 / 'photos').glob('*.jpg')}
    r = road_sel[road_sel.file_name.isin(have_now)].assign(**{'class': '터널 없음', 'portal_log_ratio': np.nan})
    ph = pd.concat([ph, r[[c for c in ph.columns if c in r.columns]]], ignore_index=True)
print(ph['class'].value_counts().to_dict())
ph['t0'] = ph.captured_at.dt.floor('h')
ph['lat_r'], ph['lon_r'] = ph.lat.round(2), ph.lon.round(2)
ph['file'] = [f'{a:.2f}_{b:.2f}_{t:%Y%m%d%H}.txt' for a, b, t in zip(ph.lat_r, ph.lon_r, ph.t0)]
utc = lambda x: (x - pd.Timedelta(hours=9)).strftime('%Y%m%d%H%M')
ph['tm1'] = [utc(x - pd.Timedelta(hours=1)) for x in ph.t0]
ph['tm2'] = [utc(x + pd.Timedelta(hours=2)) for x in ph.t0]
def fetch(r):
    params = {'int': '30', 'tm1': r.tm1, 'tm2': r.tm2, 'lat': f'{r.lat_r:.4f}', 'lon': f'{r.lon_r:.4f}', 'authKey': AUTH_KEY}
    res = requests.get(BASE_URL, params=params, timeout=60); res.raise_for_status(); return res.text
print('요청할 것', ph.file.nunique(), '번')

In [ ]:
# 먼저 시험 1번: 숫자가 나오면 다음 셀
r = next(ph.drop_duplicates('file').itertuples()); print(r.file); print(fetch(r)[:600])

In [ ]:
fails = 0
for r in ph.drop_duplicates('file').itertuples():
    f = CACHE / r.file
    if f.exists() and f.stat().st_size > 0: continue
    try: txt = fetch(r)
    except Exception as e: txt = None; print('오류', r.file, e)
    if not txt or 'error' in txt[:200].lower() or '인증' in txt[:200]:
        fails += 1
        if fails >= 5: print('연속 실패 5번 → 멈춤 (한도면 내일 다시)'); break
        continue
    fails = 0; f.write_text(txt, encoding='utf-8'); time.sleep(0.2)
print('받은 파일', sum((CACHE / x).exists() for x in ph.file.unique()), '/', ph.file.nunique())

In [ ]:
def parse(txt):
    lines = [l for l in txt.splitlines() if l.strip().startswith('|')]
    head = [x.strip() for x in lines[0].strip('|').split('|')]
    out = []
    for l in lines[1:]:
        d = dict(zip(head, [x.strip() for x in l.strip('|').split('|')]))
        for k in head[4:]:
            out.append({'time_kst': pd.to_datetime(k, format='%Y%m%d%H%M') + pd.Timedelta(hours=9),
                        'gk2a_icsr': pd.to_numeric(d[k], errors='coerce')})
    return out
g = []
for f in ph.file.unique():
    p = CACHE / f
    if p.exists():
        for x in parse(p.read_text(encoding='utf-8')): x['file'] = f; g.append(x)
g = pd.DataFrame(g).dropna().drop_duplicates(['file', 'time_kst'])
ph['tm'] = ph.captured_at.dt.ceil('h')
ph = ph.merge(g, left_on=['file', 'tm'], right_on=['file', 'time_kst'], how='left').drop(columns='time_kst')

st = pd.read_csv(SD / 'data' / '1_원본' / 'asos_stations.csv', encoding='utf-8-sig').rename(columns={'지점': 'stn_id', '위도': 'slat', '경도': 'slon'})
aso = pd.read_csv(SD / 'data' / '1_원본' / 'asos_hourly_all.csv', usecols=['stn_id', 'tm', 'icsr', 'dc10Tca'], parse_dates=['tm'])
st = st[st.stn_id.isin(aso.stn_id.unique())].reset_index(drop=True)
def nearest(lat, lon):
    d = np.hypot(st.slat - lat, (st.slon - lon) * math.cos(math.radians(lat))); return st.stn_id[d.idxmin()]
ph['stn_id'] = [nearest(a, b) for a, b in zip(ph.lat, ph.lon)]
ph = ph.merge(aso.rename(columns={'icsr': 'asos_icsr', 'dc10Tca': 'cloud'}), on=['stn_id', 'tm'], how='left')
ph['E'] = ph.gk2a_icsr.fillna(ph.asos_icsr)
ph['E_출처'] = np.where(ph.gk2a_icsr.notna(), '천리안', np.where(ph.asos_icsr.notna(), 'ASOS', '없음'))
print(ph.E_출처.value_counts().to_dict())
print('천리안 vs ASOS 상관', round(ph.gk2a_icsr.corr(ph.asos_icsr), 3))
print('구름 분포(맑음 라벨인데 실제로는?)', ph.cloud.describe().round(1).to_dict())

## 6. 저장 → 07 노트북으로
07은 `photos.csv`의 `file_name, day, time, lat, lon, E, cloud, class, scene`을 읽어요.

In [ ]:
out = ph[['file_name', 'source_zip', 'road', 'day', 'time', 'lat', 'lon', 'scene', 'class', 'portal_log_ratio',
          'gk2a_icsr', 'asos_icsr', 'cloud', 'E', 'E_출처']]
out = out[out['class'].isin(['입구', '출구', '터널 없음'])]      # 터널 없음 = 학습에만
out.to_csv(A71 / 'photos.csv', index=False, encoding='utf-8-sig')
print('저장', len(out), '장 ·', out.scene.nunique(), '장면 |', out['class'].value_counts().to_dict())